# Notebook to compare two boundary defs to see how things have changed.

In [2]:
# Import Required Libraries
import pandas as pd
import geopandas as gpd

#load centres A and centres B
centres_a = gpd.read_parquet("../data/output_files/mergeafterfiltthresh_retailcen_foursquare_mrc_2_mri_1_mrn_10_mtf_6_splitthresh_1500_merge_y/4_finalcentres_las.parquet")
# centres_b = gpd.read_parquet("../data/output_files/aug2025_foursquare_mrc_2_mri_1_mrn_10_mtf_6_splitthresh_1500_merge_y/4_finalcentres_las.parquet")
centres_b = gpd.read_parquet("../data/output_files/aug2025_2020refcut_foursquare_mrc_2_mri_1_mrn_10_mtf_6_splitthresh_1500_merge_y/4_finalcentres_las.parquet")

old_centres = gpd.read_parquet("../data/old_retail_centres/old_retail_centers.parquet")


In [3]:
#Print some basic stats about the centres
print(f"Centres A: {centres_a.shape[0]} rows")
print(f"Centres B: {centres_b.shape[0]} rows")
print(f"Old Centres: {old_centres.shape[0]} rows")

#print some comparison stats using the geometry intersections
print(f"Centres A and B intersection: {gpd.overlay(centres_a, centres_b, how='intersection').shape[0]} rows")
print(f"Centres A and Old Centres intersection: {gpd.overlay(centres_a, old_centres, how='intersection').shape[0]} rows")
print(f"Centres B and Old Centres intersection: {gpd.overlay(centres_b, old_centres, how='intersection').shape[0]} rows")

# helper function: count how many in A don't intersect anything in B
def count_not_in(A, B):
    return A[~A.geometry.apply(lambda g: B.intersects(g).any())].shape[0]

print(f"Old Centres not in Centres A: {count_not_in(old_centres, centres_a)} rows")
print(f"Old Centres not in Centres B: {count_not_in(old_centres, centres_b)} rows")

print(f"Centres A not in Old Centres: {count_not_in(centres_a, old_centres)} rows")
print(f"Centres B not in Old Centres: {count_not_in(centres_b, old_centres)} rows")

#centres in A not in B
print(f"Centres A not in Centres B: {count_not_in(centres_a, centres_b)} rows")
print(f"Centres B not in Centres A: {count_not_in(centres_b, centres_a)} rows")


Centres A: 10887 rows
Centres B: 7247 rows
Old Centres: 6423 rows


/tmp/ipykernel_473074/874018236.py:7: UserWarning: `keep_geom_type=True` in overlay resulted in 469 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  print(f"Centres A and B intersection: {gpd.overlay(centres_a, centres_b, how='intersection').shape[0]} rows")


Centres A and B intersection: 7440 rows
Centres A and Old Centres intersection: 6375 rows
Centres B and Old Centres intersection: 5744 rows
Old Centres not in Centres A: 381 rows
Old Centres not in Centres B: 938 rows
Centres A not in Old Centres: 5409 rows
Centres B not in Old Centres: 2195 rows
Centres A not in Centres B: 3861 rows
Centres B not in Centres A: 40 rows


In [5]:

# --- Old centres not intersecting centres_a ---
joined_a = gpd.sjoin(old_centres, centres_a, how="left", predicate="intersects")
old_centres_not_in_a = joined_a[joined_a.index_right.isna()].drop(columns="index_right")

# --- Old centres not intersecting centres_b ---
joined_b = gpd.sjoin(old_centres, centres_b, how="left", predicate="intersects")
old_centres_not_in_b = joined_b[joined_b.index_right.isna()].drop(columns="index_right")

# --- centres_a not intersecting centres_b ---
joined_a_b = gpd.sjoin(centres_a, centres_b, how="left", predicate="intersects")
centres_a_not_in_b = joined_a_b[joined_a_b.index_right.isna()].drop(columns="index_right")

# --- centres_b not intersecting centres_a ---
joined_b_a = gpd.sjoin(centres_b, centres_a, how="left", predicate="intersects")
centres_b_not_in_a = joined_b_a[joined_b_a.index_right.isna()].drop(columns="index_right")

# Save results
old_centres_not_in_a.to_parquet("../data/mappingoutputs/old_centres_not_in_a.parquet")
old_centres_not_in_b.to_parquet("../data/mappingoutputs/old_centres_not_in_b.parquet")
centres_a_not_in_b.to_parquet("../data/mappingoutputs/centres_a_not_in_b.parquet")
centres_b_not_in_a.to_parquet("../data/mappingoutputs/centres_b_not_in_a.parquet")
